# 지면 데이터셋 전처리_Local

`C:/Dataset/Train2YOLO_Outdoor_Raw`의 지면 polygon 라벨만 사용해 지면 전용 YOLO-seg 데이터셋을 만듭니다.

생성 결과:

```text
C:/Dataset/Train2YOLO_Outdoor_SurfaceGuideSeg/
├─ images/train
├─ images/val
├─ labels/train
├─ labels/val
├─ visualizations
└─ data.yaml
```

라벨은 189번 실외 지면 분류 흐름에 맞춰 다음 7개로 통합합니다.

```text
sidewalk
braille_guide_blocks
roadway
alley
crosswalk
bike_lane
caution_zone
```

파손 노면은 별도 클래스로 빼지 않고 원래 지면 클래스로 통합합니다.


In [1]:
from pathlib import Path
from collections import Counter
import os
import shutil
import yaml

RAW_ROOT = Path("C:/Dataset/Train2YOLO_Outdoor_Raw")
RAW_DATA_YAML = RAW_ROOT / "raw_data.yaml"
SURFACE_ROOT = Path("C:/Dataset/Train2YOLO_Outdoor_SurfaceGuideSeg")
SURFACE_DATA_YAML = SURFACE_ROOT / "data.yaml"

OVERWRITE_LABELS = True
VIS_SAMPLES_PER_SPLIT = 20

if not RAW_DATA_YAML.exists():
    raise FileNotFoundError(RAW_DATA_YAML)

print("RAW_ROOT:", RAW_ROOT)
print("SURFACE_ROOT:", SURFACE_ROOT)


RAW_ROOT: C:\Dataset\Train2YOLO_Outdoor_Raw
SURFACE_ROOT: C:\Dataset\Train2YOLO_Outdoor_SurfaceGuideSeg


## 라벨 통합 기준

In [2]:
raw_data = yaml.safe_load(RAW_DATA_YAML.read_text(encoding="utf-8"))
raw_names = raw_data["names"]
if isinstance(raw_names, dict):
    raw_names = [raw_names[i] for i in sorted(raw_names)]

CLASS_NAMES = [
    "sidewalk",
    "braille_guide_blocks",
    "roadway",
    "alley",
    "crosswalk",
    "bike_lane",
    "caution_zone",
]

RAW_TO_SURFACE = {
    "outdoor_surface__sidewalk__blocks": "sidewalk",
    "outdoor_surface__sidewalk__soil_stone": "sidewalk",
    "outdoor_surface__sidewalk__cement": "sidewalk",
    "outdoor_surface__sidewalk__damaged": "sidewalk",
    "outdoor_surface__sidewalk__other": "sidewalk",
    "outdoor_surface__sidewalk__urethane": "sidewalk",
    "outdoor_surface__sidewalk__asphalt": "sidewalk",

    "outdoor_surface__braille_guide_blocks__normal": "braille_guide_blocks",
    "outdoor_surface__braille_guide_blocks__damaged": "braille_guide_blocks",

    "outdoor_surface__roadway__normal": "roadway",

    "outdoor_surface__alley__normal": "alley",
    "outdoor_surface__alley__speed_bump": "alley",
    "outdoor_surface__alley__damaged": "alley",
    "outdoor_surface__alley": "alley",
    "outdoor_surface__alley__blocks": "alley",

    "outdoor_surface__roadway__crosswalk": "crosswalk",
    "outdoor_surface__alley__crosswalk": "crosswalk",

    "outdoor_surface__bike_lane": "bike_lane",

    "outdoor_surface__caution_zone__tree_zone": "caution_zone",
    "outdoor_surface__caution_zone__manhole": "caution_zone",
    "outdoor_surface__caution_zone__stairs": "caution_zone",
    "outdoor_surface__caution_zone__grating": "caution_zone",
    "outdoor_surface__caution_zone__repair_zone": "caution_zone",
}

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
RAW_ID_TO_NEW_ID = {}
for raw_name, merged_name in RAW_TO_SURFACE.items():
    if raw_name not in raw_names:
        print("Raw 데이터에 없는 라벨:", raw_name)
    else:
        RAW_ID_TO_NEW_ID[raw_names.index(raw_name)] = CLASS_TO_ID[merged_name]

print("최종 클래스")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i:02d}: {name}")

print("\nRaw -> Surface 매핑")
for raw_id, new_id in sorted(RAW_ID_TO_NEW_ID.items()):
    print(f"{raw_id:02d} {raw_names[raw_id]} -> {new_id:02d} {CLASS_NAMES[new_id]}")


최종 클래스
00: sidewalk
01: braille_guide_blocks
02: roadway
03: alley
04: crosswalk
05: bike_lane
06: caution_zone

Raw -> Surface 매핑
00 outdoor_surface__alley__normal -> 03 alley
01 outdoor_surface__roadway__normal -> 02 roadway
02 outdoor_surface__sidewalk__blocks -> 00 sidewalk
03 outdoor_surface__bike_lane -> 05 bike_lane
04 outdoor_surface__caution_zone__tree_zone -> 06 caution_zone
05 outdoor_surface__caution_zone__manhole -> 06 caution_zone
06 outdoor_surface__caution_zone__stairs -> 06 caution_zone
07 outdoor_surface__sidewalk__soil_stone -> 00 sidewalk
08 outdoor_surface__caution_zone__grating -> 06 caution_zone
09 outdoor_surface__braille_guide_blocks__normal -> 01 braille_guide_blocks
10 outdoor_surface__alley__speed_bump -> 03 alley
11 outdoor_surface__sidewalk__cement -> 00 sidewalk
12 outdoor_surface__sidewalk__damaged -> 00 sidewalk
13 outdoor_surface__sidewalk__other -> 00 sidewalk
14 outdoor_surface__sidewalk__urethane -> 00 sidewalk
15 outdoor_surface__sidewalk__asphalt 

## 데이터셋 생성

In [3]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)


def link_or_copy(src, dst):
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return
    if dst.exists():
        dst.unlink()
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def parse_label_line(line):
    parts = line.strip().split()
    if len(parts) < 7:
        return None, []
    cls_id = int(float(parts[0]))
    values = [float(x) for x in parts[1:]]
    return cls_id, values


def clamp01(v):
    return max(0.0, min(1.0, v))


def normalize_polygon(values):
    if len(values) < 6 or len(values) % 2 != 0:
        return None
    values = [clamp01(v) for v in values]
    xs = values[0::2]
    ys = values[1::2]
    if max(xs) - min(xs) <= 0 or max(ys) - min(ys) <= 0:
        return None
    return values


def format_seg_line(cls_id, values):
    return " ".join([str(cls_id)] + [f"{v:.6f}" for v in values])


def convert_split(split):
    src_img_dir = RAW_ROOT / "images" / split
    src_lbl_dir = RAW_ROOT / "labels" / split
    dst_img_dir = SURFACE_ROOT / "images" / split
    dst_lbl_dir = SURFACE_ROOT / "labels" / split
    ensure_dir(dst_img_dir)
    ensure_dir(dst_lbl_dir)

    images = sorted(p for p in src_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(images)
    stats = Counter()
    label_counts = Counter()

    for idx, img_path in enumerate(images, 1):
        if idx % 1000 == 0 or idx == total:
            print(f"{split}: {idx}/{total} ({idx / max(total, 1) * 100:.1f}%)")

        src_label = src_lbl_dir / f"{img_path.stem}.txt"
        if not src_label.exists():
            stats["missing_label"] += 1
            continue

        out_lines = []
        for line in src_label.read_text(encoding="utf-8").splitlines():
            raw_id, values = parse_label_line(line)
            if raw_id is None or raw_id not in RAW_ID_TO_NEW_ID:
                continue
            polygon = normalize_polygon(values)
            if polygon is None:
                stats["invalid_polygon"] += 1
                continue
            new_id = RAW_ID_TO_NEW_ID[raw_id]
            out_lines.append(format_seg_line(new_id, polygon))
            label_counts[new_id] += 1

        if out_lines:
            link_or_copy(img_path, dst_img_dir / img_path.name)
            dst_label = dst_lbl_dir / f"{img_path.stem}.txt"
            if OVERWRITE_LABELS or not dst_label.exists():
                dst_label.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
            stats["surface_images"] += 1
        else:
            stats["empty_surface"] += 1

    return stats, label_counts

all_stats = {}
total_counts = Counter()
for split in ["train", "val"]:
    stats, counts = convert_split(split)
    all_stats[split] = stats
    total_counts.update(counts)

print("\n변환 통계")
for split, stats in all_stats.items():
    print(split, dict(stats))

print("\n클래스별 polygon 수")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i:02d} {name}: {total_counts[i]}")


train: 1000/125096 (0.8%)
train: 2000/125096 (1.6%)
train: 3000/125096 (2.4%)
train: 4000/125096 (3.2%)
train: 5000/125096 (4.0%)
train: 6000/125096 (4.8%)
train: 7000/125096 (5.6%)
train: 8000/125096 (6.4%)
train: 9000/125096 (7.2%)
train: 10000/125096 (8.0%)
train: 11000/125096 (8.8%)
train: 12000/125096 (9.6%)
train: 13000/125096 (10.4%)
train: 14000/125096 (11.2%)
train: 15000/125096 (12.0%)
train: 16000/125096 (12.8%)
train: 17000/125096 (13.6%)
train: 18000/125096 (14.4%)
train: 19000/125096 (15.2%)
train: 20000/125096 (16.0%)
train: 21000/125096 (16.8%)
train: 22000/125096 (17.6%)
train: 23000/125096 (18.4%)
train: 24000/125096 (19.2%)
train: 25000/125096 (20.0%)
train: 26000/125096 (20.8%)
train: 27000/125096 (21.6%)
train: 28000/125096 (22.4%)
train: 29000/125096 (23.2%)
train: 30000/125096 (24.0%)
train: 31000/125096 (24.8%)
train: 32000/125096 (25.6%)
train: 33000/125096 (26.4%)
train: 34000/125096 (27.2%)
train: 35000/125096 (28.0%)
train: 36000/125096 (28.8%)
train: 37000/

## data.yaml 생성

In [4]:
data_yaml = {
    "path": str(SURFACE_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}
SURFACE_DATA_YAML.write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")

print(SURFACE_DATA_YAML.read_text(encoding="utf-8"))
for rel in ["images/train", "images/val", "labels/train", "labels/val"]:
    folder = SURFACE_ROOT / rel
    pattern = "*" if rel.startswith("images") else "*.txt"
    print(rel, len(list(folder.glob(pattern))) if folder.exists() else "MISSING")


path: C:\Dataset\Train2YOLO_Outdoor_SurfaceGuideSeg
train: images/train
val: images/val
names:
  0: sidewalk
  1: braille_guide_blocks
  2: roadway
  3: alley
  4: crosswalk
  5: bike_lane
  6: caution_zone

images/train 41740
images/val 4656
labels/train 41740
labels/val 4656


## 샘플 시각화

In [5]:
if VIS_SAMPLES_PER_SPLIT > 0:
    from PIL import Image, ImageDraw

    vis_dir = SURFACE_ROOT / "visualizations"
    ensure_dir(vis_dir)
    colors = ["red", "lime", "cyan", "yellow", "magenta", "orange", "white"]

    for split in ["train", "val"]:
        img_dir = SURFACE_ROOT / "images" / split
        lbl_dir = SURFACE_ROOT / "labels" / split
        samples = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)[:VIS_SAMPLES_PER_SPLIT]
        for img_path in samples:
            label_path = lbl_dir / f"{img_path.stem}.txt"
            if not label_path.exists():
                continue
            img = Image.open(img_path).convert("RGB")
            draw = ImageDraw.Draw(img)
            w, h = img.size
            for line in label_path.read_text(encoding="utf-8").splitlines():
                cls_id, values = parse_label_line(line)
                if cls_id is None:
                    continue
                pts = [(values[i] * w, values[i + 1] * h) for i in range(0, len(values), 2)]
                color = colors[cls_id % len(colors)]
                draw.line(pts + [pts[0]], fill=color, width=2)
                draw.text(pts[0], CLASS_NAMES[cls_id], fill=color)
            img.save(vis_dir / f"{split}_{img_path.name}")

    print("시각화 저장:", vis_dir)


시각화 저장: C:\Dataset\Train2YOLO_Outdoor_SurfaceGuideSeg\visualizations


## Colab 업로드용 압축

학습은 Colab에서 진행하므로 생성된 데이터셋을 zip으로 압축합니다.


In [6]:
MAKE_ZIP = True
ZIP_PATH = SURFACE_ROOT.with_suffix(".zip")

if MAKE_ZIP:
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    shutil.make_archive(str(SURFACE_ROOT), "zip", root_dir=SURFACE_ROOT)
    print("zip 생성:", ZIP_PATH)
    print("크기 GB:", round(ZIP_PATH.stat().st_size / (1024 ** 3), 3))


zip 생성: C:\Dataset\Train2YOLO_Outdoor_SurfaceGuideSeg.zip
크기 GB: 1.656
